# Tutorial 2: Training

This tutorial covers training the deepSCENIC model to learn gene regulatory networks from multiome data.

## Setup

First, let's import the required packages and check GPU availability:

In [2]:
import torch

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

deepSCENIC version: 0.1.0
Using device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


```{note}
Training deepSCENIC requires a GPU with at least 16-32GB VRAM due to the Enformer model. Training on CPU is technically possible but extremely slow (hours vs minutes per epoch). See the "Custom Sequence Models" section below if you need a lighter alternative.
```

## Load Preprocessed Data

We'll use the preprocessed MuData from Tutorial 1. The data must contain:
- RNA and ATAC modalities with matching cells
- TF annotations (`is_tf` column in `rna.var`)
- Region-to-gene search space (`r2g` in `uns`)
- Train/test split (`split` column in `obs` and `var`)

In [3]:
import os

# Set your data directory
data_dir = "../../../../data/deepSCENIC/MM_lines/"
assert os.path.exists(data_dir), "Data directory not found"

# Load preprocessed MuData
mdata = ds.read(data_dir + "preprocessed_data.h5mu")
print(
    f"Cells: {mdata.n_obs} ({(mdata.obs['split'] == 'train').sum()} train, {(mdata.obs['split'] == 'test').sum()} test)"
)
print(f"Genes: {mdata['rna'].n_vars} ({mdata['rna'].var['is_tf'].sum()} TFs)")
print(f"Regions: {mdata['atac'].n_vars}")
mdata

deepscenic.io - INFO - Loaded ../../../../data/deepSCENIC/MM_lines/preprocessed_data.h5mu: 936 cells, 10421 genes, 281820 regions


Cells: 936 (748 train, 188 test)
Genes: 10421 (1288 TFs)
Regions: 281820


MuData object with n_obs × n_vars = 936 × 292241
  obs:	'split'
  uns:	'r2g'
  2 modalities
    rna:	936 x 10421
      var:	'n_cells', 'is_tf', 'chromosome', 'tss', 'split'
      uns:	'log1p'
      layers:	'log_norm'
    atac:	936 x 281820
      var:	'chromosome', 'start', 'end', 'split'

## Register Genome

deepSCENIC extracts DNA sequences from genomic regions to learn TF binding patterns. Before training, you must register a genome FASTA file that matches your data's reference assembly.

In [10]:
try:
    genome = ds.get_genome()
    print(f"Genome already registered: {genome.name}")
except RuntimeError:
    ds.register_genome("../../../../data/genomes/hg38.fa")

Genome already registered: hg38


## Understanding Training Configuration

Training is configured through the `ds.tl.train()` function, which accepts both direct parameters and a `TrainingConfig` object. Understanding the key parameters helps you adapt training to your specific dataset and hardware.

### Two Ways to Configure Training

1. **Direct parameters** (simple): Pass common settings directly to `ds.tl.train()`
2. **TrainingConfig** (advanced): Use a config object for full control over all settings

When you provide both, **direct parameters override config values**. This lets you use a config as a baseline and tweak specific settings.

### Key Parameters

The most important parameters to adjust:

| Parameter | Default | When to change |
|-----------|---------|----------------|
| `epochs` | 100 | Increase for better convergence (500-1000 for production) |
| `batch_size` | 64 | Reduce if GPU memory errors |
| `lr` | 1e-4 | Lower if training unstable |

For the full parameter list, see {func}`~deepscenic.tl.train`.

```{tip}
The `seq_batch_size` (default: 1000) controls how many DNA sequences are processed per batch. This can be much larger than `batch_size` since sequences are processed independently.
```

## Train the Model

Now we're ready to train. The `ds.tl.train()` function initializes all model components, computes the initial E1 matrix from DNA sequences, and runs the training loop.

```{note}
The first training run downloads the pretrained Enformer model (~1GB). Subsequent runs use the cached version. Initial E1 computation also takes several minutes as it processes all genomic sequences.
```

In [ ]:
# Train the deepSCENIC model
# For demonstration we use 100 epochs; for production use 500-1000
model = ds.tl.train(
    mdata,
    epochs=100,
    batch_size=64,
    lr=1e-4,
    device=device,
)

print(f"\nTraining complete!")
print(f"Model contains {len(model.tf_names)} TFs, {len(model.gene_names)} genes, {len(model.region_names)} regions")

## Fine-tuning E2 Matrix (Strongly Recommended)

After main training, you should fine-tune the E2 (region→gene) matrix to improve generalization. This two-phase process tests the model's ability to generalize across cells and chromosomes:

- **Phase 2** (`cell_split="test"`): Fine-tune E2 on held-out cells with training features. This validates that the learned regulatory relationships generalize to unseen cells.
- **Phase 3** (`feature_split="test"`): Fine-tune E2 on training cells with held-out chromosomes. This validates generalization to unseen genomic regions.

See {func}`~deepscenic.tl.finetune_e2` for full parameter reference.

In [ ]:
# Phase 2: Fine-tune E2 on test cells
model = ds.tl.finetune_e2(
    model,
    mdata,
    cell_split="test",      # Use held-out cells
    feature_split="train",  # Use training features
    epochs=500,
    reinit_e2=True,         # Reset E2 weights before finetuning
)

# Phase 3: Fine-tune E2 on test chromosomes
model = ds.tl.finetune_e2(
    model,
    mdata,
    cell_split="train",     # Use training cells
    feature_split="test",   # Use held-out chromosomes
    epochs=500,
    reinit_e2=True,
)

## Monitor Training Progress

deepSCENIC supports logging backends for tracking training metrics:

```python
# TensorBoard logging
model = ds.tl.train(
    mdata,
    epochs=100,
    logger="tensorboard",
    log_dir="./runs/deepscenic_experiment",
    device=device,
)
```

Then launch TensorBoard in a separate terminal:
```bash
tensorboard --logdir ./runs
```

Alternatively, use Weights & Biases with `logger="wandb"`.

## Save and Load Models

Trained models can be saved for later use. The saved file includes:
- All model weights (VAE, TF2rNet, sequence model)
- The E1 matrix cache
- Training configuration
- Feature names (TFs, genes, regions)

```{note}
Model files are large (~1GB+) when using Enformer. Models trained with custom sequence models will be smaller.
```

In [ ]:
# Save the trained model
model.save(data_dir + "deepscenic_model.pt")
print(f"Model saved to {data_dir}deepscenic_model.pt")

In [ ]:
# Load a saved model in future sessions
loaded_model = ds.tl.load_model(data_dir + "deepscenic_model.pt", device=device)
print(f"Loaded model with {len(loaded_model.tf_names)} TFs")

## Custom Sequence Models

By default, deepSCENIC uses Enformer as its sequence encoder. However, you can provide your own PyTorch module instead. This is useful when:

- You want a lighter model that doesn't require 16GB+ VRAM
- You have a custom architecture tailored to your genomic context
- You want to experiment with different sequence representations

### Model Requirements

Your custom sequence model must satisfy these requirements:

| Requirement | Details |
|-------------|---------|
| **Input shape** | `(batch, seq_len, 4)` - one-hot encoded DNA sequences |
| **Output shape** | `(batch, bottleneck_size * emb_len)` - flattened embeddings |
| **Config sync** | Set matching `seq_len`, `bottleneck_size`, `emb_len` in config |

The three key parameters control the model's interface:

- **`seq_len`**: Input DNA sequence length in base pairs. Default is 640 for Enformer compatibility. Set this to match your model's expected input length.
- **`bottleneck_size`**: Embedding dimension per position. Default is 3072 (Enformer's embedding size). This is the number of features your model outputs per position.
- **`emb_len`**: Number of output positions. Default is 5 (Enformer's target_length). Combined with bottleneck_size, this determines your model's total output: `bottleneck_size * emb_len`.

In [ ]:
import torch.nn as nn

from deepscenic.tl import TrainingConfig


# Example: A lightweight convolutional sequence model
class LightweightSequenceModel(nn.Module):
    """Lightweight sequence encoder for reduced GPU memory."""

    def __init__(self, seq_len=640, bottleneck_size=256, emb_len=60):
        super().__init__()
        # Store dimensions for verification
        self.seq_len = seq_len
        self.bottleneck_size = bottleneck_size
        self.emb_len = emb_len
        self.output_dim = bottleneck_size * emb_len

        # Simple conv architecture
        self.conv = nn.Sequential(
            nn.Conv1d(4, 64, kernel_size=15, padding=7),
            nn.ReLU(),
            nn.Conv1d(64, bottleneck_size, kernel_size=15, padding=7),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(emb_len)

    def forward(self, x):
        """Transform one-hot DNA to flattened embeddings."""
        # x: (batch, seq_len, 4)
        x = x.transpose(1, 2)  # (batch, 4, seq_len)
        x = self.conv(x)  # (batch, bottleneck_size, seq_len)
        x = self.pool(x)  # (batch, bottleneck_size, emb_len)
        return x.flatten(1)  # (batch, bottleneck_size * emb_len)


# Create model instance
custom_model = LightweightSequenceModel(
    seq_len=640,
    bottleneck_size=256,
    emb_len=60,
)

# Configure training with matching dimensions
config = TrainingConfig(
    sequence_model=custom_model,
    seq_len=640,  # Must match model's expected input
    bottleneck_size=256,  # Must match model's output channels
    emb_len=60,  # Must match model's output length
    epochs=100,
)

print(f"Custom model output: {custom_model.output_dim} = {config.bottleneck_size} x {config.emb_len}")

### Training with Custom Models

```python
# Train with custom sequence model
model = ds.tl.train(mdata, config=config, device=device)

# Save the model
model.save("custom_model.pt")
```

### Loading Custom Models

When loading a model trained with a custom sequence model, you must provide an instance of the same architecture:

```python
# Recreate the model architecture (same as during training)
custom_model = LightweightSequenceModel(seq_len=640, bottleneck_size=256, emb_len=60)

# Load with the custom model
loaded_model = ds.tl.load_model(
    "custom_model.pt",
    device=device,
    sequence_model=custom_model,  # Required for custom models
)
```

```{warning}
The model architecture must exactly match what was used during training. Only the weights are saved in the checkpoint file, not the model class definition.
```

## Complete Workflow Example

Here's a complete training workflow with all optional stages:

In [ ]:
# Complete deepSCENIC training workflow
#
# # 1. Register genome
# ds.register_genome("path/to/hg38.fa")
#
# # 2. Load preprocessed data
# mdata = ds.read("preprocessed_data.h5mu")
#
# # 3. Main training
# model = ds.tl.train(
#     mdata,
#     epochs=500,
#     batch_size=64,
#     lr=1e-4,
#     device="cuda",
# )
#
# # 4. Fine-tune E2 on test cells (strongly recommended)
# model = ds.tl.finetune_e2(model, mdata, cell_split="test", epochs=500)
#
# # 5. Fine-tune E2 on test chromosomes (strongly recommended)
# model = ds.tl.finetune_e2(model, mdata, feature_split="test", epochs=500)
#
# # 6. Save final model
# model.save("final_model.pt")

## Next Steps

Now that you have a trained model, you can:

- Diagnose model training and inspect results: {doc}`03_model_diagnosis`
- Extract and analyze the learned GRN: {doc}`04_grn_analysis`
- Simulate perturbation experiments: {doc}`05_perturbation_analysis`